# 03 - Ground Truth and Curated Datasets

This notebook turns individual evaluation examples into a small, repeatable test suite. The expected behavior is often called **ground truth**: reference information that helps an evaluator decide whether the agent handled a scenario well. Each scenario contains one or more user messages plus clear expectations for the answer and tool use.

You will:

1. Describe good behavior with expected responses, assertions, and tool sequences.
2. Build scenarios with one turn or several connected turns.
3. Run a small dataset against the deployed CityAnalyst agent.
4. Read scenario status, evaluator scores, and explanations.

**Estimated time:** 40-60 minutes  
**Creates AWS resources:** No persistent resources. Running the dataset invokes the agent and judge models and can incur cost.  
**Feature status:** Dataset evaluation is public preview as of August 14, 2026.

## 1. Three kinds of reference input

Reference inputs describe what a successful scenario should look like. They guide evaluation without requiring one exact sentence or one exact conversation transcript.

| Reference | Best for | Example |
|---|---|---|
| `expected_response` | An example of the facts or meaning a correct answer should contain | Seattle population and land area |
| `assertions` | Plain-language requirements that allow several valid answers | The response must identify both cities |
| `expected_trajectory` | The tool calls expected for the task, in order | `lookup_city`, then `compare_cities` |

A trajectory is the path of tool calls taken during a conversation. The expected trajectory tells an evaluator which tools a successful run should use and in what order. Evaluator availability can vary, so use the evaluator IDs shown by your installed CLI.

## 2. Load the checked-in scenario file

The dataset file uses JSONL, which means each line contains one complete JSON scenario. The managed dataset schema names the response field `expectedResponse`, while the Python SDK uses `expected_response`. The loader below translates between those two names.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from bedrock_agentcore.evaluation import Dataset, PredefinedScenario, Turn

DATASET_PATH = Path("data/city_scenarios.jsonl")
raw_scenarios = [
    json.loads(line)
    for line in DATASET_PATH.read_text().splitlines()
    if line.strip()
]

scenarios = []
for item in raw_scenarios:
    turns = [
        Turn(
            input=turn["input"],
            expected_response=turn.get("expectedResponse"),
        )
        for turn in item["turns"]
    ]
    scenarios.append(
        PredefinedScenario(
            scenario_id=item["scenario_id"],
            turns=turns,
            assertions=item.get("assertions"),
            expected_trajectory=item.get("expected_trajectory"),
        )
    )

dataset = Dataset(scenarios=scenarios)
scenario_catalog = pd.DataFrame(
    [
        {
            "Scenario": item.scenario_id,
            "Turns": len(item.turns),
            "First user message": item.turns[0].input,
            "Expected tools": item.expected_trajectory or [],
        }
        for item in scenarios
    ]
)
display(scenario_catalog)

## 3. Inspect the multi-turn case

A multi-turn scenario keeps several user messages in the same session. The shared session ID lets the agent connect a follow-up such as "compare it with Portland" to the Seattle request from the previous turn. The dataset runner creates and reuses that session ID automatically.

In [ ]:
multi_turn = next(
    scenario
    for scenario in scenarios
    if scenario.scenario_id == "multi-turn-follow-up"
)
display(
    pd.DataFrame(
        [
            {
                "Turn": index,
                "User message": turn.input,
                "Expected response": turn.expected_response,
            }
            for index, turn in enumerate(multi_turn.turns, start=1)
        ]
    )
)
print("Scenario requirements:", multi_turn.assertions)
print("Expected tool sequence:", multi_turn.expected_trajectory)

## 4. Add reference information to one session

You do not need to create a dataset for every investigation. If you already have a session to review, `ReferenceInputs` lets you attach ground truth directly to that evaluation: an expected answer, a list of requirements, and an expected tool sequence. This is a quick way to study one problem before deciding whether it belongs in the reusable dataset.

In [ ]:
from datetime import timedelta

from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs
from src.workshop_utils import (
    RuntimeInvoker,
    load_runtime_info,
    make_session_id,
    wait_for_session_trace,
)

runtime = load_runtime_info()
invoker = RuntimeInvoker(runtime)
direct_session_id = make_session_id("ground-truth")
direct_response = invoker.invoke(
    "Compare Seattle, WA with Portland, OR. Which is denser?",
    direct_session_id,
)
wait_for_session_trace(direct_session_id)
display(Markdown("**CityAnalyst response**"))
display(direct_response)

direct_results = EvaluationClient(region_name=runtime.region).run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolSelectionAccuracy",
    ],
    session_id=direct_session_id,
    agent_id=runtime.runtime_id,
    look_back_time=timedelta(hours=1),
    reference_inputs=ReferenceInputs(
        expected_response=(
            "Seattle is more populous and denser than Portland "
            "in the workshop dataset."
        ),
        assertions=[
            "The response compares both requested cities.",
            "The response identifies Seattle as denser.",
        ],
        expected_trajectory=["compare_cities"],
    ),
)

direct_rows = []
for result in direct_results:
    context = result.get("context", {})
    if context.get("spanId"):
        target_type, target_id = "Tool call", context["spanId"]
    elif context.get("traceId"):
        target_type, target_id = "Trace", context["traceId"]
    else:
        target_type = "Session"
        target_id = context.get("sessionId", direct_session_id)

    direct_rows.append(
        {
            "Evaluator": result.get("evaluatorId") or "Unknown evaluator",
            "Target": target_type,
            "Target ID": target_id,
            "Score": result.get("value"),
            "Label": result.get("label"),
            "Explanation": result.get("explanation"),
            "Error": result.get("errorMessage") or result.get("errorCode"),
        }
    )

direct_results_df = pd.DataFrame(direct_rows)
display(direct_results_df[["Evaluator", "Target", "Score", "Label", "Error"]])
for row in direct_rows:
    detail = row["Error"] or row["Explanation"] or "No explanation was returned."
    display(
        Markdown(
            f"**{row['Evaluator']}** - {row['Target']} `{row['Target ID']}`  \n"
            f"Score: `{row['Score']}` | Label: `{row['Label'] or 'Not available'}`\n\n"
            f"{detail}"
        )
    )

## 5. Configure the dataset runner

The dataset runner automates three steps for every scenario:

1. Send each scenario's user messages to CityAnalyst.
2. Wait until the corresponding trace data appears in CloudWatch.
3. Send the recorded spans and reference inputs to the selected evaluators.

`CloudWatchAgentSpanCollector` checks repeatedly until the spans arrive, up to the configured timeout. Because the collector already handles that wait, `evaluation_delay_seconds=0` avoids adding a second fixed delay.

In [ ]:
from bedrock_agentcore.evaluation import (
    CloudWatchAgentSpanCollector,
    EvaluationRunConfig,
    EvaluatorConfig,
    OnDemandEvaluationDatasetRunner,
)

collector = CloudWatchAgentSpanCollector(
    log_group_name=runtime.log_group_name,
    region=runtime.region,
    max_wait_seconds=180,
    poll_interval_seconds=10,
)

run_config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(
        evaluator_ids=[
            "Builtin.GoalSuccessRate",
            "Builtin.Correctness",
            "Builtin.Helpfulness",
            "Builtin.ToolSelectionAccuracy",
        ]
    ),
    evaluation_delay_seconds=0,
    max_concurrent_scenarios=3,
)

runner = OnDemandEvaluationDatasetRunner(region=runtime.region)

## 6. Start with three scenarios

Start with three scenarios. This gives you quick feedback and limits agent and judge model calls while you confirm that the workflow is configured correctly.

In [ ]:
small_dataset = Dataset(scenarios=scenarios[:3])
evaluation_result = runner.run(
    config=run_config,
    dataset=small_dataset,
    agent_invoker=invoker,
    span_collector=collector,
)

scenario_status_df = pd.DataFrame(
    [
        {
            "Scenario": result.scenario_id,
            "Status": result.status,
            "Session ID": result.session_id,
            "Evaluator groups": len(result.evaluator_results),
            "Error": result.error,
        }
        for result in evaluation_result.scenario_results
    ]
)
display(scenario_status_df)

In [ ]:
def collect_dataset_rows(result):
    rows = []
    for scenario_result in result.scenario_results:
        if scenario_result.status != "COMPLETED":
            rows.append(
                {
                    "scenario": scenario_result.scenario_id,
                    "evaluator": "Scenario execution",
                    "target_type": "Scenario",
                    "target_id": scenario_result.session_id,
                    "value": None,
                    "label": "ERROR",
                    "explanation": scenario_result.error,
                    "error_code": "SCENARIO_ERROR",
                    "error_message": scenario_result.error,
                }
            )
            continue

        for evaluator_result in scenario_result.evaluator_results:
            for evaluator_score in evaluator_result.results:
                context = evaluator_score.get("context") or {}
                if context.get("spanId"):
                    target_type, target_id = "Tool call", context["spanId"]
                elif context.get("traceId"):
                    target_type, target_id = "Trace", context["traceId"]
                else:
                    target_type = "Session"
                    target_id = context.get(
                        "sessionId", scenario_result.session_id
                    )

                error_code = evaluator_score.get("errorCode")
                error_message = evaluator_score.get("errorMessage")
                rows.append(
                    {
                        "scenario": scenario_result.scenario_id,
                        "evaluator": evaluator_result.evaluator_id,
                        "target_type": target_type,
                        "target_id": target_id,
                        "value": evaluator_score.get("value"),
                        "label": (
                            "ERROR"
                            if error_code or error_message
                            else evaluator_score.get("label")
                        ),
                        "explanation": evaluator_score.get("explanation"),
                        "error_code": error_code,
                        "error_message": error_message,
                    }
                )
    return rows


def show_dataset_results(result):
    rows = collect_dataset_rows(result)
    results_df = pd.DataFrame(rows)
    if results_df.empty:
        print("No evaluation results were returned.")
        return results_df

    display(
        results_df[
            [
                "scenario",
                "evaluator",
                "target_type",
                "value",
                "label",
                "error_message",
            ]
        ]
    )

    current_group = None
    for row in rows:
        group = (row["scenario"], row["evaluator"])
        if group != current_group:
            current_group = group
            display(Markdown(f"### {row['scenario']} - {row['evaluator']}"))

        detail = (
            row["error_message"]
            or row["error_code"]
            or row["explanation"]
            or "No explanation was returned."
        )
        detail_label = (
            "Error"
            if row["error_message"] or row["error_code"]
            else "Explanation"
        )
        display(
            Markdown(
                f"**{row['target_type']}:** `{row['target_id']}`  \n"
                f"**Score:** `{row['value'] if row['value'] is not None else 'Not available'}` | "
                f"**Label:** `{row['label'] or 'Not available'}`\n\n"
                f"**{detail_label}:** {detail}"
            )
        )
    return results_df


dataset_results_df = show_dataset_results(evaluation_result)
if dataset_results_df.empty:
    raise RuntimeError("The dataset run returned no evaluation results.")
evaluation_errors = dataset_results_df[
    dataset_results_df["error_code"].notna()
    | dataset_results_df["error_message"].notna()
]
if not evaluation_errors.empty:
    raise RuntimeError(
        f"{len(evaluation_errors)} scenario or evaluator result(s) failed. "
        "Review the error rows and explanations above before continuing."
    )

## 7. Run the full dataset when ready

The complete dataset adds a few useful edge cases:

- a greeting that should not call a tool
- an unknown city that should produce a clear "not found" answer
- a two-turn follow-up that checks whether the agent remembers the first turn

Run the full set after the first three scenarios complete successfully. This keeps early troubleshooting quick and inexpensive.

In [ ]:
RUN_FULL_DATASET = False

if RUN_FULL_DATASET:
    full_result = runner.run(
        config=run_config,
        dataset=dataset,
        agent_invoker=invoker,
        span_collector=collector,
    )
    full_status_df = pd.DataFrame(
        [
            {
                "Scenario": result.scenario_id,
                "Status": result.status,
                "Session ID": result.session_id,
                "Error": result.error,
            }
            for result in full_result.scenario_results
        ]
    )
    display(full_status_df)
    full_results_df = show_dataset_results(full_result)
else:
    print("Set RUN_FULL_DATASET=True after the first three scenarios succeed.")

## 8. Managed datasets through the CLI

A managed dataset stores the scenarios as an AgentCore resource so a team can refer to the same dataset by name. The cell below keeps the three actions separate: add the dataset to the local project, deploy it to AWS, and run an evaluation.

Use the Python runner when you want direct control in code, such as inside a test suite. Use a managed dataset when several people or automated workflows need to share a named, versioned set of scenarios.

In [ ]:
import shutil

from src.workshop_utils import MODULE_ROOT, run_cli, run_cli_json

MANAGED_DATASET_NAME = "CityRegression"
CREATE_MANAGED_DATASET = False
DEPLOY_MANAGED_DATASET = False
RUN_MANAGED_DATASET_EVAL = False

config_path = MODULE_ROOT / "agentcore" / "agentcore.json"
project_config = json.loads(config_path.read_text())
configured_datasets = {
    item["name"] for item in project_config.get("datasets", [])
}
managed_dataset_path = (
    MODULE_ROOT / "agentcore" / "datasets" / f"{MANAGED_DATASET_NAME}.jsonl"
)

if CREATE_MANAGED_DATASET:
    if MANAGED_DATASET_NAME not in configured_datasets:
        result = run_cli(
            "add",
            "dataset",
            "--name",
            MANAGED_DATASET_NAME,
            "--schema-type",
            "AGENTCORE_EVALUATION_PREDEFINED_V1",
            "--description",
            "Stable city-agent regression scenarios",
        )
        print(result.stdout.strip())
    else:
        print(f"{MANAGED_DATASET_NAME} is already present in agentcore.json.")
    managed_dataset_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(MODULE_ROOT / "data" / "city_scenarios.jsonl", managed_dataset_path)
    print(f"Prepared {managed_dataset_path.relative_to(MODULE_ROOT)}")
else:
    print("Set CREATE_MANAGED_DATASET=True to add the managed dataset locally.")

if DEPLOY_MANAGED_DATASET:
    if not managed_dataset_path.exists():
        raise RuntimeError("Create and populate the managed dataset before deploying it.")
    diff = run_cli("deploy", "--target", "default", "--diff")
    print(diff.stdout.strip())
    managed_deploy_result = run_cli_json(
        "deploy", "--target", "default", "--yes"
    )
    print(f"Deployed {MANAGED_DATASET_NAME} to the default target.")

if RUN_MANAGED_DATASET_EVAL:
    managed_eval_result = run_cli_json(
        "run",
        "eval",
        "--runtime",
        "CityAnalyst",
        "--dataset",
        MANAGED_DATASET_NAME,
        "--evaluator",
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
    )
    managed_run = managed_eval_result.get("run", managed_eval_result)
    managed_summary = []
    managed_details = []
    for evaluator_result in managed_run.get("results", []):
        scores = evaluator_result.get("sessionScores", [])
        managed_summary.append(
            {
                "Evaluator": evaluator_result.get("evaluator"),
                "Aggregate score": evaluator_result.get("aggregateScore"),
                "Sessions scored": len(scores),
                "Errors": sum(
                    bool(item.get("errorCode") or item.get("errorMessage"))
                    for item in scores
                ),
            }
        )
        for score in scores:
            managed_details.append(
                {
                    "evaluator": evaluator_result.get("evaluator"),
                    "session": score.get("sessionId", "Not available"),
                    "value": score.get("value"),
                    "label": score.get("label"),
                    "detail": (
                        score.get("errorMessage")
                        or score.get("errorCode")
                        or score.get("explanation")
                        or "No explanation was returned."
                    ),
                }
            )

    if managed_summary:
        display(pd.DataFrame(managed_summary))
    else:
        print("No managed evaluation results were returned.")
    for item in managed_details:
        display(
            Markdown(
                f"**{item['evaluator']}** - session `{item['session']}`  \n"
                f"Score: `{item['value'] if item['value'] is not None else 'Not available'}` | "
                f"Label: `{item['label'] or 'Not available'}`\n\n"
                f"{item['detail']}"
            )
        )

## 9. Dataset design checklist

A useful regression dataset should include:

- common successful tasks
- failures you have seen or expect to see
- no-tool requests
- invalid or missing parameters
- requests for data the agent does not have
- multi-turn follow-ups
- scenarios that distinguish similar tools

Keep each expectation clear and review dataset changes alongside code changes. A small, well-understood suite is usually more useful than a large collection of ambiguous examples.

## 10. Checkpoint

You now have:

- reference information for one existing session
- a repeatable dataset runner
- single-turn and multi-turn scenarios
- a runner that waits for trace data before evaluating

Continue to [04 - Custom Evaluators](04-custom-evaluators.ipynb) to measure a requirement that is specific to your own application.